# SoccerNet GSR — Colab Inference

Runs the SoccerNet Game State Reconstruction baseline on Colab with GPU.

**Setup:** Runtime → Change runtime type → T4 GPU

### Pipeline stages

| # | Stage | What it does | Colab |
|---|-------|-------------|-------|
| 1 | `bbox_detector` | YOLOv11 — detect player & ball bounding boxes | ✅ |
| 2 | `reid` | PRTReid — extract appearance embeddings for re-ID | ✅ |
| 3 | `track` | StrongSORT + BPBreid — multi-object tracking | ✅ |
| 4 | `pitch` | NBW calibration — detect pitch lines for homography | ✅ |
| 5 | `calibration` | NBW calibration — pixel-to-pitch coordinate mapping | ✅ |
| 6 | `jersey_number_detect` | MMOCR — read jersey numbers from player crops | ❌ Skipped |
| 7 | `tracklet_agg` | Voting + role assignment (GK, outfield) | ✅ |
| 8 | `team` | K-means clustering on embeddings → 2 teams | ✅ |
| 9 | `team_side` | Mean position → which team attacks left/right | ✅ |

> **Jersey number detection** is skipped because MMOCR depends on mmcv, which has no pre-built wheels for Colab's Python 3.12 + torch 2.x. All other stages work normally.

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"

## Setup — Install dependencies

This cell:
1. Clones our repo + sn-gamestate
2. Patches sn-gamestate's version pins for Colab compatibility (Python 3.12, torch 2.x)
3. Installs all dependencies **except** mmcv/mmdet/mmocr (jersey number detection not available on Colab)

In [ ]:
import os

# === Step 1: Clone repos ===
REPO_URL = "https://github.com/Moiz005/SoccerVision-Player-Tracking-3D-Reconstruction.git"
REPO_NAME = "SoccerVision-Player-Tracking-3D-Reconstruction"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

if not os.path.exists("sn-gamestate"):
    !git clone https://github.com/SoccerNet/sn-gamestate.git

# === Step 2: Patch sn-gamestate's pyproject.toml ===
# sn-gamestate pins torch==1.13.1 and requires-python>=3.9,<3.10
# Colab has Python 3.12 + torch 2.x — we relax these pins
pyproject_path = "sn-gamestate/pyproject.toml"
with open(pyproject_path, "r") as f:
    content = f.read()

# Allow Python 3.10+
content = content.replace(
    'requires-python = ">=3.9,<3.10"',
    'requires-python = ">=3.9"'
)

# Remove torch version pin (let it use Colab's torch 2.x)
content = content.replace(
    '    "torch==1.13.1",',
    '    "torch",'
)

# Float numpy for Python 3.12 compat
content = content.replace(
    '    "numpy==1.26.4",',
    '    "numpy>=1.26.4",'
)

with open(pyproject_path, "w") as f:
    f.write(content)

print("Patched pyproject.toml: relaxed requires-python, torch pin, numpy pin")

# === Step 3: Install sn-gamestate without resolving deps ===
# --no-deps prevents pip from trying to install torch 1.13.1 or other conflicting versions
%cd sn-gamestate
!pip install --no-deps -e .

# === Step 4: Install git dependencies (skipped by --no-deps) ===
!pip install "prtreid @ git+https://github.com/VlSomers/prtreid"
!pip install "torchreid @ git+https://github.com/VlSomers/bpbreid"

# Install the calibration plugin bundled with sn-gamestate
# NOTE: calibration plugin requires Python <3.10 but works on 3.12; --ignore-requires-python bypasses this
!pip install --no-deps -e plugins/calibration --ignore-requires-python

# === Step 5: Install remaining PyPI deps ===
# NOT installed: mmcv, mmdet, mmocr (no pre-built wheels for Colab Python 3.12 + torch 2.x)
# This means jersey number detection (stage 6) is unavailable
!pip install "tracklab==1.3.24" \
    "soccernet==0.1.55" \
    "lightning==2.0.9" \
    "transformers==4.35.2" \
    "tokenizers==0.15.2" \
    "easyocr==1.7.1"

print("\n=== Dependencies installed ===")
print("Available stages: bbox_detector, reid, track, pitch, calibration, tracklet_agg, team, team_side")
print("Skipped stages:   jersey_number_detect (requires mmcv/mmocr)")

# === Step 6: Compatibility patches ===

# Fix 1: albumentations version conflict (prtreid needs <2.0 for functional import)
# Pin to 1.3.1 to avoid pydantic version clash with tracklab's lightning==2.0.9 (needs pydantic<2.2.0)
!pip install "albumentations==1.3.1" -q
print("albumentations pinned to 1.3.1 (avoids pydantic conflict)")

# Fix 2: prtreid torch.load weights_only issue (PyTorch 2.6+ defaults to True)
TORCHTOOLS = "/usr/local/lib/python3.12/dist-packages/prtreid/utils/torchtools.py"
with open(TORCHTOOLS) as f:
    content = f.read()
content = content.replace(
    "checkpoint = torch.load(fpath, map_location=map_location)",
    "checkpoint = torch.load(fpath, map_location=map_location, weights_only=False)"
)
with open(TORCHTOOLS, "w") as f:
    f.write(content)
print("prtreid torchtools.py patched (weights_only=False)")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

import tracklab
print(f"TrackLab: {getattr(tracklab, '__version__', 'imported OK')}")

import sn_gamestate
print("sn-gamestate: OK")

# Verify mmcv is NOT installed (expected)
try:
    import mmcv
    print(f"mmcv: {mmcv.__version__}")
except ImportError:
    print("mmcv: not installed (expected — jersey detection unavailable)")

print("\nAll core dependencies verified!")

## Download Dataset

Downloads the SoccerNet GSR validation split (~11GB). This only needs to run once — Colab persists files within a session.

Dataset structure after extraction:
```
data/SoccerNetGS/
  gamestate-2024/
    SNGS-021/
      img1/          # Frame images (000001.jpg, 000002.jpg, ...)
      Labels-GameState.json
    SNGS-022/
      ...
```

In [ ]:
import os, zipfile, glob

DATA_DIR = f"/content/{REPO_NAME}/sn-gamestate/data/SoccerNetGS"
os.makedirs(DATA_DIR, exist_ok=True)

# Download validation split
from SoccerNet.Downloader import SoccerNetDownloader
dl = SoccerNetDownloader(LocalDirectory=DATA_DIR)
dl.downloadDataTask(task="gamestate-2024", split=["valid"])

# Extract all zip files (SoccerNet places them in gamestate-2024/ subdirectory)
zips = glob.glob(os.path.join(DATA_DIR, "**/*.zip"), recursive=True)
for zip_path in zips:
    print(f"Extracting {os.path.basename(zip_path)}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)
    print(f"  Done")

# Verify structure
gs_dir = os.path.join(DATA_DIR, "gamestate-2024")
if os.path.exists(gs_dir):
    videos = [d for d in os.listdir(gs_dir) if d.startswith("SNGS-")]
    print(f"\nDataset ready: {len(videos)} videos")
    # Show structure of first video
    if videos:
        sample = os.path.join(gs_dir, videos[0])
        print(f"  Example: {videos[0]}/")
        for item in os.listdir(sample):
            path = os.path.join(sample, item)
            if os.path.isdir(path):
                print(f"    {item}/ ({len(os.listdir(path))} files)")
            else:
                print(f"    {item}")
else:
    print("\nWARNING: gamestate-2024/ folder not found after extraction")

## Run Baseline on Official Validation Video

Runs the pipeline on **1 validation video** from SoccerNet (limited to first 4 frames for quick testing).

The pipeline executes 8 stages (jersey number detection excluded):
1. **bbox_detector** — YOLOv11 detects players + ball
2. **reid** — PRTReid extracts appearance embeddings
3. **track** — StrongSORT tracks players across frames
4. **pitch** — NBW calibration finds pitch lines
5. **calibration** — Maps pixels to real-world pitch coordinates
6. **tracklet_agg** — Aggregates tracklets, assigns roles (GK/outfield)
7. **team** — K-means clusters players into 2 teams
8. **team_side** — Determines left/right attacking direction

Change `dataset.nframes=4` to `dataset.nframes=-1` to run on all frames.
Model weights auto-download on first run (~2GB).

In [ ]:
import os, subprocess, shutil
os.chdir(f"/content/{REPO_NAME}/sn-gamestate")

# Build the dataset path in Python (avoids shell variable expansion issues)
# SNGS-XXX folders are directly under SoccerNetGS/ (zip extracted here)
DATASET_PATH = f"/content/{REPO_NAME}/sn-gamestate/data/SoccerNetGS"
print(f"Dataset path: {DATASET_PATH}")
print(f"Exists: {os.path.exists(DATASET_PATH)}")

# Locate the tracklab CLI
tracklab_cmd = shutil.which("tracklab")
print(f"tracklab binary: {tracklab_cmd}")

# Run 8-stage pipeline on 1 video (jersey_number_detect excluded)
# Using subprocess.run() instead of ! shell to capture full stdout/stderr
# --config-dir points to sn-gamestate's configs (plugin entry point may not register with --no-deps)
# ~modules.jersey_number_detect = Hydra override to remove this module
# pipeline= explicitly lists the 8 stages to run (order matters)
# dataset.dataset_path = points to the downloaded frames
# dataset.nframes = limit to first N frames (set -1 for all frames)
result = subprocess.run(
    [tracklab_cmd, "-cn", "soccernet",
     "--config-dir", "sn_gamestate/configs",
     "~modules.jersey_number_detect",
     "pipeline=[bbox_detector,reid,track,pitch,calibration,tracklet_agg,team,team_side]",
     f"dataset.dataset_path={DATASET_PATH}",
     "dataset.nframes=4"],
    capture_output=True, text=True, timeout=300
)

print("\n" + "="*60)
print("STDOUT:")
print("="*60)
print(result.stdout)
print("\n" + "="*60)
print("STDERR (last 5000 chars):")
print("="*60)
print(result.stderr[-5000:] if len(result.stderr) > 5000 else result.stderr)
print("\n" + "="*60)
print(f"Return code: {result.returncode}")
print("="*60)

## View Results

The pipeline generates:
- **Annotated video** — bounding boxes, player IDs, pitch overlay
- **Tracker state** (.pklz) — full tracking data for re-analysis
- **Evaluation metrics** — GS-HOTA score (official SoccerNet benchmark)

In [ ]:
import glob
from IPython.display import Video, display

output_videos = glob.glob(
    "output/**/visualization/videos/*.mp4",
    recursive=True
)

print(f"Found {len(output_videos)} output video(s):")
for v in output_videos:
    print(f"  - {v}")

if output_videos:
    print(f"\nPlaying: {output_videos[0]}")
    display(Video(output_videos[0], width=800))
else:
    print("No output videos found. Check the pipeline output above for errors.")

## Next Steps

If this ran successfully, the baseline works on official data.

**Next:** Run this on your own football clip → Phase 4 (Custom Video Adapter)